# 08 · ¿La brecha se achica con ocupación más granular? (CASEN, CIUO 4 dígitos)

## La pregunta

Los notebooks 06-07 controlaron por **grupo ocupacional a 1 dígito** (CIUO, ~9 categorías amplias: "Profesionales", "Técnicos nivel medio", etc. — lo máximo que permite la ESI). Dentro de "Profesionales" caben médicos, abogados, ingenieros y periodistas al mismo tiempo — categorías con salarios y composición de género muy distintos. **Si controláramos por la ocupación exacta (médico vs. enfermero vs. técnico en enfermería), ¿se reduciría la brecha "no explicada"?**

CASEN permite responder esto: tiene `oficio4_08`, el código CIUO a **4 dígitos** (cientos de ocupaciones específicas), algo que la ESI no ofrece.

## Alcance: por qué solo 2022 y 2024

CASEN 2017 codifica la ocupación con la revisión **CIUO-88** (`oficio4_88`); 2022 y 2024 usan **CIUO-08** (`oficio4_08`) — son clasificaciones distintas, no una simple renumeración (a diferencia del caso de área de formación del proyecto `empleabilidad-formacion-casen-chile`). Para no mezclar dos sistemas de códigos distintos, este notebook usa **solo 2022 y 2024**, ambos en CIUO-08.

In [1]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import patsy
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.05)
pd.set_option('display.float_format', '{:.2f}'.format)
os.makedirs('outputs/figures', exist_ok=True)

RUTA_CASEN = '../../CASEN'
COLUMNAS = ['sexo','edad','e6a','o10','oficio4_08','ytrabajocor','expr','activ','varunit']

frames = []
for anio in [2022, 2024]:
    df = pd.read_stata(f'{RUTA_CASEN}/casen_{anio}.dta', columns=COLUMNAS, convert_categoricals=False)
    df['anio'] = anio
    frames.append(df)
panel = pd.concat(frames, ignore_index=True)
print(f'Panel CASEN 2022+2024: {len(panel):,} personas')

Panel CASEN 2022+2024: 420,598 personas


## 1. Primer vistazo: ¿persiste la brecha dentro de la MISMA ocupación de 4 dígitos?

In [2]:
ocupados = panel[(panel['activ']==1) & (panel['ytrabajocor'].notna()) & (panel['ytrabajocor']>0) &
                  (panel['oficio4_08'].notna())].copy()
print(f'Ocupados con ingreso y ocupación válida: {len(ocupados):,}')
print(f'Códigos de ocupación (4 dígitos) distintos: {ocupados["oficio4_08"].nunique()}')

EJEMPLOS = {2211:'Médicos generales', 2212:'Médicos especialistas',
            2221:'Enfermeros profesionales', 3221:'Técnicos/auxiliares de enfermería'}

filas = []
for cod, nombre in EJEMPLOS.items():
    sub = ocupados[ocupados['oficio4_08']==cod]
    fila = {'Ocupación (CIUO-08)': f'{nombre} ({cod})', 'n': len(sub)}
    for sx, lab in [(1,'Hombres'), (2,'Mujeres')]:
        s = sub[sub['sexo']==sx]
        fila[f'Ingreso {lab}'] = np.average(s['ytrabajocor'], weights=s['expr']) if len(s) else np.nan
    fila['Brecha (%)'] = (fila['Ingreso Mujeres']/fila['Ingreso Hombres'] - 1) * 100
    filas.append(fila)

print(pd.DataFrame(filas).round(0).to_string(index=False))

Ocupados con ingreso y ocupación válida: 176,542
Códigos de ocupación (4 dígitos) distintos: 444
                     Ocupación (CIUO-08)    n  Ingreso Hombres  Ingreso Mujeres  Brecha (%)
                Médicos generales (2211)  343       3164642.00       2792769.00      -12.00
            Médicos especialistas (2212)  388       4894792.00       3897019.00      -20.00
         Enfermeros profesionales (2221) 1142       1513088.00       1460705.00       -3.00
Técnicos/auxiliares de enfermería (3221) 2599        773912.00        669538.00      -13.00


### Interpretación

Incluso comparando exactamente la misma ocupación de 4 dígitos, la brecha **no desaparece** — pero su magnitud varía mucho según la ocupación específica. Eso ya sugiere que agrupar a 1 dígito ("Profesionales") mezcla ocupaciones con brechas muy distintas entre sí.

## 2. La prueba real: mismo dato, misma regresión, dos niveles de detalle ocupacional

In [3]:
NIVEL_EDUC = {
    1:'Básica',2:'Básica',3:'Básica',4:'Básica',5:'Básica',6:'Básica',7:'Básica',
    8:'Media',9:'Media',10:'Media',11:'Media',
    12:'Técnica sup.',13:'Universitaria',14:'Posgrado',15:'Posgrado',
}

muestra = ocupados.copy()
muestra['mujer'] = (muestra['sexo']==2).astype(int)
muestra['nivel_grp'] = muestra['e6a'].map(NIVEL_EDUC)
muestra['log_ingreso'] = np.log(muestra['ytrabajocor'])
muestra['edad2'] = muestra['edad']**2
muestra['ciuo_1digito'] = (muestra['oficio4_08'] // 1000).astype(int).astype(str)  # primer dígito CIUO-08

# Ocupaciones granulares con muestra suficiente (n>=30); el resto se agrupa como 'Otras'
conteo_ocup = muestra['oficio4_08'].value_counts()
ocup_validas = set(conteo_ocup[conteo_ocup>=30].index)
muestra['oficio4_grp'] = muestra['oficio4_08'].apply(lambda c: str(int(c)) if c in ocup_validas else 'otras')
muestra['cluster_id'] = muestra['anio'].astype(str) + '_' + muestra['varunit'].astype(str)

muestra = muestra.dropna(subset=['nivel_grp','o10'])
print(f'Muestra final para la regresión: {len(muestra):,}')
print(f'Categorías de ocupación granular (n>=30): {len(ocup_validas)} + "otras"')
print(f'Categorías de ocupación amplia (1 dígito CIUO): {muestra["ciuo_1digito"].nunique()}')

Muestra final para la regresión: 176,542
Categorías de ocupación granular (n>=30): 356 + "otras"
Categorías de ocupación amplia (1 dígito CIUO): 11


In [4]:
formula_base = ('log_ingreso ~ mujer + edad + edad2 + C(nivel_grp) + o10 + C(anio)')

modelo_amplio = smf.wls(formula_base + ' + C(ciuo_1digito)', data=muestra, weights=muestra['expr']).fit(
    cov_type='cluster', cov_kwds={'groups': muestra['cluster_id']})

modelo_granular = smf.wls(formula_base + ' + C(oficio4_grp)', data=muestra, weights=muestra['expr']).fit(
    cov_type='cluster', cov_kwds={'groups': muestra['cluster_id']})

def resumen(modelo, nombre):
    coef = modelo.params['mujer']
    ic = modelo.conf_int().loc['mujer']
    brecha = (np.exp(coef)-1)*100
    print(f'{nombre}')
    print(f'  Brecha ajustada: {brecha:.1f}%  (IC95%: {(np.exp(ic[0])-1)*100:.1f}% a {(np.exp(ic[1])-1)*100:.1f}%)  '
          f'p={modelo.pvalues["mujer"]:.2e}  R²={modelo.rsquared:.3f}')
    return brecha

b_amplio = resumen(modelo_amplio, 'Control por ocupación AMPLIA (1 dígito CIUO, ~9 categorías — equivalente a lo que permite la ESI)')
print()
b_granular = resumen(modelo_granular, f'Control por ocupación GRANULAR (4 dígitos CIUO, {len(ocup_validas)} categorías)')
print()
print(f'Diferencia: {b_granular - b_amplio:+.1f} puntos porcentuales')

Control por ocupación AMPLIA (1 dígito CIUO, ~9 categorías — equivalente a lo que permite la ESI)
  Brecha ajustada: -24.4%  (IC95%: -25.2% a -23.6%)  p=0.00e+00  R²=0.449

Control por ocupación GRANULAR (4 dígitos CIUO, 356 categorías)
  Brecha ajustada: -17.6%  (IC95%: -18.5% a -16.7%)  p=6.73e-252  R²=0.508

Diferencia: +6.8 puntos porcentuales


### Interpretación

Esta es la comparación limpia que responde la pregunta: **mismos datos, misma especificación, la única diferencia es el nivel de detalle de la ocupación**. El resultado confirma la hipótesis: al pasar de 1 dígito (9 categorías) a 4 dígitos (356 categorías), la brecha ajustada se reduce de **-24.4% a -17.6%** — una caída de **6.8 puntos porcentuales**.

Esto significa que una parte real de lo que el control amplio dejaba como "no explicado" era en realidad **segregación ocupacional fina**: dentro de categorías como "Profesionales" los hombres se concentran más en las especialidades mejor pagadas (ej. médicos especialistas por sobre enfermería), y el 1 dígito no puede distinguir eso. Aun así, **-17.6% es una brecha grande que sigue sin explicarse** ni siquiera controlando por la ocupación exacta — la segregación fina explica una porción real, pero no la mayoría, de la brecha total.

## 3. ¿Dónde se concentra la brecha residual? Top ocupaciones con mayor y menor brecha (n≥50)

In [5]:
resultados_ocup = []
for cod in ocup_validas:
    sub = muestra[muestra['oficio4_08']==cod]
    if len(sub) < 50:
        continue
    h = sub[sub['mujer']==0]
    m = sub[sub['mujer']==1]
    if len(h)<10 or len(m)<10:
        continue
    ing_h = np.average(h['ytrabajocor'], weights=h['expr'])
    ing_m = np.average(m['ytrabajocor'], weights=m['expr'])
    resultados_ocup.append({'oficio4_08': int(cod), 'n': len(sub), 'n_h': len(h), 'n_m': len(m),
                             'brecha_pct': (ing_m/ing_h - 1)*100})

df_ocup = pd.DataFrame(resultados_ocup).sort_values('brecha_pct')
print('=== 10 ocupaciones (n≥50, con ambos sexos representados) con mayor brecha en contra de mujeres ===')
print(df_ocup.head(10).to_string(index=False))
print()
print('=== 10 ocupaciones con menor brecha o favorables a mujeres ===')
print(df_ocup.tail(10).to_string(index=False))

=== 10 ocupaciones (n≥50, con ambos sexos representados) con mayor brecha en contra de mujeres ===
 oficio4_08   n  n_h  n_m  brecha_pct
       5142 685   25  660      -67.15
       5311 951   16  935      -62.70
       6122 106   76   30      -56.34
       1219  62   42   20      -55.29
       1411 144   51   93      -54.73
       7514  59   23   36      -54.23
       6130 243  203   40      -53.46
       7318 424   65  359      -53.13
       4313  91   22   69      -51.82
       1431  75   52   23      -50.11

=== 10 ocupaciones con menor brecha o favorables a mujeres ===
 oficio4_08   n  n_h  n_m  brecha_pct
       2146 291  250   41       16.98
       1112 183  113   70       17.77
       3132 169  159   10       18.82
       2512 286  248   38       19.84
       2529  66   56   10       21.38
       7119 394  381   13       27.94
       2643  55   21   34       36.27
       2652 148  128   20       41.00
       7313 124   50   74       58.69
       3432  66   11   55      126.03


### Interpretación

Esta tabla usa códigos CIUO-08 (se pueden buscar en el clasificador oficial del INE para identificar la ocupación exacta). Es el nivel de detalle más fino que permiten datos públicos chilenos sin acceder a microdatos de la Superintendencia de Pensiones o el Servicio de Impuestos Internos — mucho más allá de lo que cualquier notebook anterior de este repositorio pudo mostrar.

**Advertencia de lectura:** varias de las brechas más extremas de esta tabla (ej. -67% o +126%) provienen de ocupaciones con **muy pocos hombres o mujeres** dentro de la categoría (ej. 25 hombres contra 660 mujeres) — con un grupo tan chico, el promedio de ingreso de ese grupo es poco estable y la brecha calculada puede ser ruido más que señal. Esta tabla sirve para explorar patrones, no para afirmaciones fuertes sobre una ocupación puntual; la comparación robusta de este notebook es la de la sección 2, que pondera todas las ocupaciones a la vez.

## Conclusiones

1. **La brecha no desaparece ni siquiera dentro de la misma ocupación de 4 dígitos** — persiste en médicos, enfermeros y técnicos por igual, aunque con magnitudes distintas
2. **Comparar controlando por ocupación amplia vs. granular, con los mismos datos**, aísla el efecto puro de la granularidad — la sección 2 cuantifica exactamente cuánto de la brecha "no explicada" de los notebooks 06-07 se debía a que el 1 dígito no distinguía especialidades dentro de "Profesionales" y categorías similares
3. **Límite de esta extensión:** no se pudo incluir 2017 (usa CIUO-88, no CIUO-08) ni comparar directamente con los coeficientes de ESI (fuente de datos, años y definición de ingreso distintos) — la comparación rigurosa es la interna de la sección 2, no un cruce entre repositorios
4. **Complementa** a `empleabilidad-formacion-casen-chile`: mismo tipo de hallazgo (agregar granularidad revela heterogeneidad oculta), ahora aplicado a la brecha salarial de género en vez de a la empleabilidad por área de formación